# Lab 10: Data File Analyzer (DS-STAR Component)## Objectifs- Implementer le module File Analyzer de DS-STAR- Analyser des fichiers heterogenes (CSV, JSON, Markdown, texte)- Extraire des metadonnees structurees

## 1. Configuration

In [1]:
import sys
sys.path.insert(0, '..')

import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
from dataclasses import dataclass, asdict

from config import get_settings
from utils import LLMClient

In [2]:
settings = get_settings()
print(f'Provider: {settings.active_provider}')

Provider: vllm


## 2. FileMetadata DataClass

In [3]:
@dataclass
class FileMetadata:
    filename: str
    format: str
    size_bytes: int
    num_rows: int = None
    num_columns: int = None
    columns: list = None
    statistics: dict = None
    sample_data: list = None

## 3. FileAnalyzer Class

In [4]:
class FileAnalyzer:
    SUPPORTED = {'.csv': 'csv', '.json': 'json', '.md': 'markdown', '.txt': 'text'}

    def __init__(self, llm_client=None):
        self.llm = llm_client or LLMClient()

    def detect_format(self, path):
        ext = Path(path).suffix.lower()
        return self.SUPPORTED.get(ext, 'unknown')

    def analyze_csv(self, path):
        df = pd.read_csv(path)
        cols = []
        for c in df.columns:
            info = {'name': c, 'dtype': str(df[c].dtype), 'missing_pct': round(df[c].isna().mean()*100, 2)}
            if df[c].dtype in ['int64', 'float64']:
                info['stats'] = {'min': float(df[c].min()), 'max': float(df[c].max()), 'mean': float(df[c].mean())}
            cols.append(info)
        return FileMetadata(
            filename=Path(path).name, format='csv', size_bytes=os.path.getsize(path),
            num_rows=len(df), num_columns=len(df.columns), columns=cols,
            sample_data=df.head(5).to_dict('records')
        )

    def analyze(self, path):
        fmt = self.detect_format(path)
        if fmt == 'csv':
            return self.analyze_csv(path)
        raise ValueError(f"Format non supporte: {fmt}")

    def generate_summary(self, meta):
        lines = [f"Fichier: {meta.filename}", f"Format: {meta.format}", f"Taille: {meta.size_bytes/1024:.1f} KB"]
        if meta.num_rows:
            lines.append(f"Lignes: {meta.num_rows}, Colonnes: {meta.num_columns}")
        if meta.columns:
            for c in meta.columns[:5]:
                lines.append(f"  - {c['name']} ({c['dtype']}): {c.get('missing_pct', 0)}% manquant")
        return "\n".join(lines)

    def generate_llm_summary(self, meta, question=None):
        ctx = self.generate_summary(meta)
        if meta.sample_data:
            ctx += f"\n\nEchantillon:\n{json.dumps(meta.sample_data[:2], indent=2, default=str)[:400]}"
        prompt = f"Analyse ce fichier et resume-le:\n{ctx}\n{f'Question: {question}' if question else ''}"
        return self.llm.generate(prompt, temperature=0.3)

## 4. Tests

In [5]:
# Creation fichier test
import tempfile
test_dir = tempfile.mkdtemp()

df = pd.DataFrame({
    'id': range(1, 101),
    'product': [f'P{i}' for i in range(1, 101)],
    'category': np.random.choice(['A', 'B', 'C'], 100),
    'price': np.random.uniform(10, 500, 100).round(2),
    'qty': np.random.randint(1, 50, 100)
})
df.loc[10:15, 'price'] = np.nan

csv_path = os.path.join(test_dir, 'products.csv')
df.to_csv(csv_path, index=False)
print(f'CSV cree: {csv_path}')

CSV cree: C:\Users\jsboi\AppData\Local\Temp\tmpqav10cx8\products.csv


In [6]:
# Test analyseur
analyzer = FileAnalyzer()
meta = analyzer.analyze(csv_path)
print(analyzer.generate_summary(meta))

Fichier: products.csv
Format: csv
Taille: 1.9 KB
Lignes: 100, Colonnes: 5
  - id (int64): 0.0% manquant
  - product (str): 0.0% manquant
  - category (str): 0.0% manquant
  - price (float64): 6.0% manquant
  - qty (int64): 0.0% manquant


In [7]:
# Colonnes detail
for c in meta.columns:
    print(f"{c['name']}: {c['dtype']}, {c.get('missing_pct', 0)}% manquant")
    if 'stats' in c:
        print(f"  -> min={c['stats']['min']:.1f}, max={c['stats']['max']:.1f}, mean={c['stats']['mean']:.1f}")

id: int64, 0.0% manquant
  -> min=1.0, max=100.0, mean=50.5
product: str, 0.0% manquant
category: str, 0.0% manquant
price: float64, 6.0% manquant
  -> min=15.2, max=493.5, mean=260.8
qty: int64, 0.0% manquant
  -> min=1.0, max=49.0, mean=24.8


## 5. Resume LLM

In [8]:
# Resume intelligent
summary = analyzer.generate_llm_summary(meta)
print(summary)



Voici une analyse et un résumé du fichier **`products.csv`** basé sur les métadonnées fournies :

### 📊 Vue d'ensemble
*   **Type de fichier :** CSV (Comma Separated Values)
*   **Taille :** 1.9 KB (fichier léger)
*   **Dimensions :** 100 lignes (enregistrements) × 5 colonnes (attributs)
*   **Sujet :** Il s'agit d'un catalogue ou d'un inventaire de produits.

### 📋 Structure des données
Le fichier contient 5 colonnes principales :
1.  **`id`** (`int64`) : Identifiant unique du produit.
2.  **`product`** (`str`) : Nom ou référence du produit.
3.  **`category`** (`str`) : Catégorie du produit (ex: B, C).
4.  **`price`** (`float64`) : Prix unitaire du produit.
5.  **`qty`** (`int64`) : Quantité en stock.

### 🛡️ Qualité des données
*   **Exhaustivité :** La majorité des colonnes sont complètes (0% de valeurs manquantes).
*   **Point d'attention :** La colonne **`price`** présente **6% de valeurs manquantes**. Cela pourrait nécessiter un nettoyage ou une imputation avant une analyse fin

In [9]:
# Resume guide
q = "Quelles analyses puis-je faire sur ces donnees?"
print(f"Question: {q}\n")
result = analyzer.generate_llm_summary(meta, q)
print(result)

Question: Quelles analyses puis-je faire sur ces donnees?





Voici une analyse détaillée et un résumé du fichier `products.csv`, ainsi que les pistes d'analyse que vous pouvez exploiter.

### 1. Résumé du Fichier

*   **Nature des données :** Catalogue ou inventaire de produits.
*   **Volume :** Petit dataset (100 lignes, 1.9 KB).
*   **Structure :** 5 colonnes (`id`, `product`, `category`, `price`, `qty`).
*   **Qualité des données :** Globalement propre, mais avec un point critique.
    *   **Intégrité :** Les colonnes `id`, `product`, `category` et `qty` sont complètes (0% de valeurs manquantes).
    *   **Problème identifié :** La colonne `price` présente **6.0% de valeurs manquantes** (soit environ 6 produits sans prix défini).

### 2. Pistes d'Analyse Possibles

Compte tenu de la structure des données, voici les analyses que vous pouvez réaliser, classées par objectif :

#### A. Analyse de la Qualité des Données (Data Quality)
*   **Traitement des valeurs manquantes :** Décider comment gérer les 6 prix manquants (supprimer les lignes, im

## 6. Resume du Lab### Points cles1. FileAnalyzer: Analyse automatique de fichiers2. Metadonnees: Extraction de schemas et stats3. Resume LLM: Contextualisation pour le Planner### Prochaine etape- Lab 11: Boucle Planner-Coder-Verifier

In [10]:
# Cleanup
import shutil
shutil.rmtree(test_dir)
print('Done')

Done


## Exercice

À vous d'étendre le FileAnalyzer pour gérer un nouveau format de fichier !


In [ ]:
# Exercice : Étendez le FileAnalyzer pour gérer les fichiers JSON
# 1. Ajoutez une méthode analyze_json
# 2. Testez-la sur un fichier JSON de test

import json

# Créons un fichier JSON de test
test_json_path = os.path.join(test_dir, 'test_data.json')
test_data = {
    'products': [
        {'id': 1, 'name': 'Laptop', 'price': 999.99, 'stock': 15},
        {'id': 2, 'name': 'Mouse', 'price': 19.99, 'stock': 50},
        {'id': 3, 'name': 'Keyboard', 'price': 49.99, 'stock': 30}
    ],
    'metadata': {
        'source': 'inventory_system',
        'last_updated': '2026-03-13'
    }
}

with open(test_json_path, 'w') as f:
    json.dump(test_data, f)

# Exercice : Implémentez la méthode analyze_json
class ExtendedFileAnalyzer(FileAnalyzer):
    def analyze_json(self, path):
        """Analyse un fichier JSON et extrait les métadonnées."""
        with open(path, 'r') as f:
            data = json.load(f)
        
        # Compte les clés au niveau racine
        root_keys = list(data.keys())
        
        # Trouve la plus grande liste
        max_list = 0
        list_name = None
        for key, value in data.items():
            if isinstance(value, list):
                if len(value) > max_list:
                    max_list = len(value)
                    list_name = key
        
        return FileMetadata(
            filename=Path(path).name,
            format='json',
            size_bytes=os.path.getsize(path),
            num_rows=max_list,
            num_columns=len(root_keys),
            columns=root_keys,
            sample_data=[root_keys, list_name]
        )

# Testez l'analyseur étendu
extended_analyzer = ExtendedFileAnalyzer()
json_meta = extended_analyzer.analyze_json(test_json_path)

print("Métadonnées du fichier JSON :")
print(f"  Fichier: {json_meta.filename}")
print(f"  Clés racine: {json_meta.columns}")
print(f"  Plus grande liste: {json_meta.sample_data[1]} ({json_meta.num_rows} éléments)")

# Nettoyage
os.remove(test_json_path)
print("\nExercice terminé avec succès !")
